# Template-subtracted photometry with AutoPhOT

This notebook runs AutoPhOT on the bundled example: a 300 s i-band image of SN 2022mop (a type-IIn supernova at redshift 0.017) taken with the SLT on 2025-01-04. The pipeline reduces the frame, aligns and subtracts a Pan-STARRS template, measures the transient on the difference image, and writes a calibrated magnitude plus the diagnostic plots shown at the end.

Two folders under `examples/` supply the inputs:

- `example_data/` holds the science frame `images.fits` and `templates/ip_template/`, a processed Pan-STARRS i-band reference.
- `autophot_db/` is the working database. `telescope.yml` maps FITS headers to instruments and bands, `tns_objects/` caches target lookups (`2022mop.yml` resolves this target), and `sn2022mop_ps1_catalog.csv` supplies the zeropoint calibrators.

Nothing here needs the network. The template is on disk, the target resolves from a cached TNS record, and the calibration catalog is the CSV of Pan-STARRS magnitudes produced when the template was reduced - so the reference and the science frame share one photometric system.

## Before you start

- AutoPhOT importable in this kernel (`pip install .` from the repository, or `conda install -c conda-forge -c astro-sean autophot`).
- SExtractor on `PATH` as `sex` or `source-extractor`. Source detection and the SFFT backend both call it; without it the run falls back to slower pure-Python detection and the subtraction drops to HOTPANTS.
- Python packages `sfft` (with `numba`), `reproject`, and `astroalign`; a `hotpants` binary on `PATH` is the fallback backend.
- Run the notebook from `examples/` - the paths below are relative to it.

## Imports and setup

`AutomatedPhotometry.load()` returns every configuration key as one nested dict. The cells below set only what this run needs; every other key keeps its default from `autophot/data/default_input.yml`, the full parameter reference.

In [ ]:
import os
import shutil
import sys
from pathlib import Path

try:
    from autophot import AutomatedPhotometry
except ModuleNotFoundError:
    # Source checkout without an install: the repo root is the parent of examples/.
    sys.path.insert(0, str(Path.cwd().parent))
    from autophot import AutomatedPhotometry

import autophot
print(f"autophot {autophot.__version__} -> {autophot.__file__}")

autophot_input = AutomatedPhotometry.load()

examples_dir = Path.cwd()
if not (examples_dir / "example_data").is_dir():
    raise FileNotFoundError(
        f"example_data not found under {examples_dir}; start the notebook from examples/"
    )

In [ ]:
# SExtractor is an external binary, not a Python package. When it lives in a
# conda environment that is not on PATH, point the run at it here.
if not (shutil.which("sex") or shutil.which("source-extractor")):
    for _bin in sorted(Path.home().glob("*conda3/envs/*/bin")):
        if (_bin / "sex").exists() or (_bin / "source-extractor").exists():
            os.environ["PATH"] = str(_bin) + os.pathsep + os.environ["PATH"]
            print(f"added {_bin} to PATH")
            break
    else:
        print("[warn] SExtractor not found; the run will use the slower Python fallback")

## 1. Point AutoPhOT at the data

| Key | Value | What it does |
|---|---|---|
| `fits_dir` | `example_data/` | Directory scanned recursively for science FITS. A `templates/` folder inside supplies references. Outputs go to a `<fits_dir>_REDUCED/` sibling. Set as an absolute path below - the pipeline changes directory while running. |
| `wdir` | `autophot_db/` | Working directory for the telescope database and the TNS and catalog caches. |
| `outdir_name` | `'REDUCED'` | Output directory suffix. |
| `target_name` | `'2022mop'` | Resolved through `<wdir>/tns_objects/2022mop.yml`, then TNS or SIMBAD when online. Use `target_ra`/`target_dec` (degrees) for an unnamed position - setting a name and coordinates together raises an error. |
| `selected_filters` | `['i']` | Bands to process; an empty list takes every band found. |
| `restart` | `False` | Skips images that already produced an `Output_*.csv`. Set `True` to force reprocessing. |
| `rng_seed` | `42` | Seed for the limiting-magnitude injection trials. |

In [ ]:
# Absolute paths - the pipeline os.chdir()s into each output directory
# while running, so relative locations would resolve against the wrong cwd.
autophot_input['fits_dir'] = str(examples_dir / 'example_data')
autophot_input['wdir'] = str(examples_dir / 'autophot_db')
autophot_input['outdir_name'] = 'REDUCED'

autophot_input['target_name'] = '2022mop'

autophot_input['selected_filters'] = ['i']
autophot_input['restart'] = False
autophot_input['rng_seed'] = 42

## 2. Calibration catalog

The zeropoint fit needs a photometric catalog. This example reads a local CSV (`custom`) rather than querying a survey: the file is the Pan-STARRS i-band calibration produced when the template itself was reduced, so the science frame and the reference share one magnitude system and the run works offline. A custom CSV needs `ra` and `dec` columns plus `<band>`/`<band>_err` magnitude pairs.

On your own field, `use_catalog='auto'` or `'panstarrs'` downloads the catalog once into `<wdir>/catalog_queries/` and later runs reuse it.

| Key | Value | What it does |
|---|---|---|
| `catalog.use_catalog` | `'custom'` | `'auto'`, a survey name, a band-to-catalog dict, or `'custom'` for a local CSV. |
| `catalog.catalog_custom_fpath` | `autophot_db/sn2022mop_ps1_catalog.csv` | CSV path required by `custom`. |

In [ ]:
autophot_input['catalog']['use_catalog'] = 'custom'
autophot_input['catalog']['catalog_custom_fpath'] = str(
    examples_dir / 'autophot_db' / 'sn2022mop_ps1_catalog.csv'
)

## 3. Astrometry

The science frame already carries a good WCS, so `redo_wcs` stays off and no solver runs. Turn it on for frames without reliable astrometry; that path needs `solve-field` from astrometry.net, or `scamp`/`swarp` with `solver='scamp'`.

In [ ]:
autophot_input['wcs']['redo_wcs'] = False

## 4. Template subtraction

Templates live under `fits_dir/templates/<band>_template/` - the bundled `ip_template/` is the legacy `i`-band folder name, still accepted. Set `download_templates='panstarrs'` to fetch a reference instead of supplying one.

| Key | Value | What it does |
|---|---|---|
| `do_subtraction` | `True` | Enable the stage. |
| `alignment_method` | `'reproject'` | Template registration: `'spalipy'` (the default, most capable), `'reproject'` (pure WCS resampling), `'astroalign'`, `'scamp'`, `'swarp'`, `'tweakwcs'`. Both frames here carry solid WCS, so resampling by WCS is enough. |
| `method` | `'sfft'` | Backend: `'sfft'`, `'hotpants'`, `'zogy'`, or `'auto'`. SFFT needs `sex` and `numba`; on failure the pipeline retries with looser flags, then falls back to HOTPANTS. |
| `forceconv` | `'REF'` | Convolution direction. `REF` keeps the science PSF on the difference image, so the science ePSF applies directly; the pipeline vetoes the direction when it would need deconvolution. |
| `sfft_backend` | `'numpy'` | `'cupy'` for a CUDA GPU. |
| `kernel_order` | `0` | Spatial polynomial degree of the matching kernel; `'auto'` raises it only when the field supports it. |

In [ ]:
autophot_input['template_subtraction']['do_subtraction'] = True
autophot_input['template_subtraction']['alignment_method'] = 'reproject'
autophot_input['template_subtraction']['method'] = 'sfft'
autophot_input['template_subtraction']['forceconv'] = 'REF'
autophot_input['template_subtraction']['sfft_backend'] = 'numpy'
autophot_input['template_subtraction']['kernel_order'] = '0'

## 5. Run

`run_photometry` reduces the template files first, then the science frames, then subtracts and measures the transient. One frame takes a couple of minutes; the return value is the aggregated output CSV. Progress prints below, and each image writes a full `LOG_<image>.log` inside its output folder. `nCPU` and `nGPU` only matter for multi-image runs.

In [ ]:
autophot_input['nCPU'] = 1
autophot_input['nGPU'] = 0
autophot_input['global_verbose_level'] = 1

loc = AutomatedPhotometry.run_photometry(
    default_input=autophot_input
)
print(loc)

## 6. The measurement

`LightCurve_Output.csv` gets one row per frame. The columns worth reading:

- `is_detection` - the pipeline's detection call, gated on S/N, fitted-profile shape, and a false-alarm probability measured on the same image.
- `mag_psf`, `mag_psf_err` - the PSF-fit magnitude on the difference image.
- `snr_psf`, `prob_detect`, `mf_snr`, `reduced_chi2` - the evidence behind the call.
- `zp_psf`, `zp_psf_err`, `zp_psf_reliable`, `zp_psf_flags` - the zeropoint and its quality flags.
- `limiting_mag`, `limiting_mag_5s2n` - injection-based limits, used when the epoch is a non-detection.

In [ ]:
import pandas as pd

if not loc:
    raise RuntimeError("run_photometry returned no output; check the log above")
lc = pd.read_csv(loc)
cols = [
    'mjd', 'filter', 'is_detection', 'mag_psf', 'mag_psf_err', 'snr_psf',
    'prob_detect', 'reduced_chi2', 'zp_psf', 'zp_psf_err', 'limiting_mag',
]
print(lc[cols].T.to_string())
print('\nZP flags:', lc['zp_psf_flags'].iloc[0])

## 7. The diagnostic plots

Every processed frame gets a folder under `example_data_REDUCED/` holding the QA plots beside the FITS products and logs. The cells below display the ones worth reading after each subtraction run.

In [ ]:
from IPython.display import Image, display

image_out = next(Path(loc).parent.glob('*/Output_*.csv')).parent

def show(pattern):
    for p in sorted(image_out.glob(pattern)):
        display(Image(filename=str(p)))

### Subtraction check

Science frame, aligned template, and difference image, each zoomed on the transient in the inset. Cyan boxes mark the sources used to match the frames; magenta hatching is masked area, mostly sky the template does not cover. In the difference panel the host galaxy has cancelled and a compact positive blob sits at the SN position - that is the measured flux. Dipoles or rings around the field stars would mean the kernel under-subtracted; here the field goes to noise apart from the target.

In [ ]:
show('Subtraction_Check_*.png')

### Zeropoint

Catalog against instrumental magnitude for the calibrators that passed the cuts; the dashed lines are the adopted slope-1 zeropoints for aperture and PSF photometry, the dotted lines the free-slope fits. Only nine calibrators survive in this small catalog, so the run flags both zeropoints unreliable (spatially correlated residuals and a slope that drifts off unity). The zeropoint still applies; a wider survey cone would tighten it.

In [ ]:
show('Zeropoint_*.png')

### Source check

The science frame with its detected sources: PSF candidates in green, catalog calibrators in blue, sources feeding the FWHM estimate in orange. An almost-empty panel means thin photometry downstream - check image quality and the detection thresholds.

In [ ]:
show('Source_Check_*.png')

### Template registration

Residual vectors of the aligned template on the science grid, one arrow per matched source, magnified. The quadrant labels give the median residual in pixels - sub-pixel everywhere here. Only a handful of sources near the target survived the matching cuts, which is why so few arrows appear. A large or coherent pattern would mean the registration is off and the subtraction will leave dipoles.

In [ ]:
show('Alignment_Vectors_*.png')

### Target PSF fit

The fitted transient cutout on the difference image (left, with the fitted centre marked) and the post-fit residual (right). A good fit leaves flat noise in the residual panel; a ring or a leftover dipole would mean the model misfit and the flux is biased. This fit gave a reduced chi-square of 1.29.

In [ ]:
show('PSF_Target_*.png')

### Limiting magnitude

How the limit is measured: the pipeline injects fake stars near the target on the difference image and records what fraction comes back. The top panel is recovery fraction against injected magnitude for the S/N >= 3 and S/N >= 5 gates, with the 50%-completeness points marked. The cutouts show injected stars at the SN position at different depths, and the maps on the right are the injection sites. This frame reports a 3-sigma limit near 20.8 mag.

In [ ]:
show('Completeness_*.png')

## What happened, and where next

SN 2022mop subtracts to a clean point source at the catalog position: i = 16.59 +/- 0.12 mag at S/N ~ 130, roughly two and a half years after discovery - a plausible light level for a SN IIn still fed by circumstellar interaction. The difference panel of the subtraction check is the evidence: the host galaxy's light cancels and only the transient remains.

To point this notebook at your own data: set `fits_dir` to your images, `target_name` to a TNS-resolvable name (or `target_ra`/`target_dec`), set `use_catalog` to `'auto'` or a survey name, and either drop a `<band>_template/` folder with a reference FITS into `fits_dir/templates/` or set `download_templates='panstarrs'`. The full parameter reference is `autophot/data/default_input.yml`.